# Where language models slip on analyst questions: a bilingual gold set and grading rubric

**Real part (PUBLIC DATA):** Eurostat and INSEE series, Wikipedia articles at fixed revisions → 120 gold tasks, each in English and French, with answers **computed and verified** (scripts, SQL, brute-force solver).
**Simulated part (SIMULATED):** 240 answers of a fictional model with planted errors, 3 simulated reviewers, guidelines v1 then v2. No real language model is called.

Workflow: write the task → compute the reference answer → write the step-by-step explanation → error taxonomy → rubric → reviewer agreement → better guidelines.

In [1]:
import json, sys, re
from pathlib import Path
import duckdb, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from puzzle_solver import solutions, violated, STOPS
P, PUB = ROOT / 'data' / 'processed', ROOT / 'data' / 'public'
tasks = pd.read_csv(P / 'tasks.csv'); answers = pd.read_csv(P / 'model_answers.csv')
graded = pd.read_csv(P / 'graded.csv').fillna({'error_code': ''})
print(tasks.type.value_counts().to_dict(), '| answers', len(answers), '| graded', len(graded))

{'series_analysis': 40, 'logic_puzzle': 30, 'claim_check': 25, 'summarise_chunk': 25} | answers 240 | graded 240


## 1. Are the reference answers right? Re-verify every task from the sources

In [2]:
con = duckdb.connect()
con.execute(f"CREATE TABLE r AS SELECT * FROM read_csv('{(PUB / 'eurostat_retail.csv').as_posix()}', types={{'time':'VARCHAR'}})")
con.execute(f"CREATE TABLE n AS SELECT * FROM read_csv('{(PUB / 'eurostat_nights.csv').as_posix()}', types={{'time':'VARCHAR'}})")
ok = {}
# growth_window: winner on 3-month averages, recomputed in SQL from the prompt's countries and months
gw = tasks[tasks.subtype == 'growth_window']
names = {'Spain':'ES','France':'FR','Germany':'DE','Italy':'IT','Netherlands':'NL','Belgium':'BE','Austria':'AT',
         'Portugal':'PT','Poland':'PL','Sweden':'SE','Ireland':'IE','Finland':'FI'}
mon = {m: i + 1 for i, m in enumerate(['January','February','March','April','May','June','July','August','September','October','November','December'])}
def ym(s):
    m, y = s.split(); return f"{y}-{mon[m]:02d}"
n_ok = 0
for _, t in gw.iterrows():
    m = re.search(r'between (\w+ \d{4}) and (\w+ \d{4}): (.*)\?', t.prompt_en)
    s, e = ym(m.group(1)), ym(m.group(2))
    geos = [names[x.strip()] for x in re.split(r', | and ', m.group(3).split('?')[0])]
    q = f'''WITH a AS (SELECT geo, avg(value) FILTER (WHERE time BETWEEN '{s}' AND strftime(CAST('{s}-01' AS DATE) + INTERVAL 2 MONTH, '%Y-%m')) AS a0,
                          avg(value) FILTER (WHERE time BETWEEN strftime(CAST('{e}-01' AS DATE) - INTERVAL 2 MONTH, '%Y-%m') AND '{e}') AS a1
             FROM r WHERE geo IN ({",".join(repr(g) for g in geos)}) GROUP BY geo)
           SELECT geo FROM a ORDER BY a1 / a0 DESC LIMIT 1'''
    n_ok += con.execute(q).fetchone()[0] == re.search(r'winner_3m_avg=(\w+)', t.check).group(1)
ok['growth_window (SQL)'] = f'{n_ok}/{len(gw)}'
# puzzles: brute force again
pz = tasks[tasks.type == 'logic_puzzle']
ok['logic_puzzle (solver)'] = f"{sum(solutions(p['n'], [tuple(x) for x in p['rules']]) == [tuple(p['solution'])] for p in pz.puzzle.map(json.loads))}/{len(pz)}"
# summaries: every number in the reference must appear in the source lead
meta = pd.read_csv(PUB / 'texts_meta.csv'); raw = ROOT / 'data' / 'raw'
if raw.exists():
    n_ok = 0; sm = tasks[tasks.subtype == 'summary']
    for _, t in sm.iterrows():
        m = meta[meta.revid.astype(str) == t.series_code.split('=')[1]].iloc[0]
        lead = re.split(r'\n==', (raw / m.raw_file).read_text(encoding='utf-8'))[0]
        n_ok += all(x in lead for x in re.findall(r'\d{3,4}', t.ref_en + ' ' + t.ref_fr))
    ok['summary (numbers in source)'] = f'{n_ok}/{len(sm)}'
ok

{'growth_window (SQL)': '10/10',
 'logic_puzzle (solver)': '30/30',
 'summary (numbers in source)': '5/5'}

The other reference answers are computed inside `src/build_series_tasks.py` from the same CSV files (year-on-year ranks, points vs %, peaks, claims) and carry their check in the `check` column. **Every reference answer is computed, none is typed by eye** (the five lead summaries are hand-written, and their figures are checked against the source).

## 2. Where does the simulated model slip?

In [3]:
acc = pd.read_csv(P / 'accuracy_by_type.csv'); acc

,task_type,answers,pass_pct,partial_pct,incorrect_pct
0,series_analysis,80,48.8,25.0,26.3
1,logic_puzzle,60,50.0,1.7,48.3
2,claim_check,50,64.0,14.0,22.0
3,summarise_chunk,50,82.0,8.0,10.0


In [4]:
pd.read_csv(P / 'error_code_totals.csv').head(13)

,error_code,answers,en,fr,share_pct,main_task_type
0,CONSTRAINT_IGNORED,25,14,11,25.5,logic_puzzle
1,PROVISIONAL_AS_FINAL,13,2,11,13.3,series_analysis
2,ARITHMETIC,11,6,5,11.2,series_analysis
3,WRONG_WINDOW,10,6,4,10.2,series_analysis
4,PCT_VS_POINTS,8,5,3,8.2,series_analysis
5,STALE_SOURCE,7,4,3,7.1,claim_check
6,HALLUCINATED_DETAIL,5,2,3,5.1,summarise_chunk
7,NUTS_LABEL,5,0,5,5.1,series_analysis
8,MISSED_BLOCK,4,3,1,4.1,summarise_chunk
9,FR_TERM,4,0,4,4.1,logic_puzzle


## 3. The events, one by one
### 3.1 Wrong comparison window (the 'which site grew the most?' trap)

In [5]:
t = tasks[(tasks.subtype == 'growth_window') & tasks.check.str.contains('winner_endpoint')].iloc[0]
print(t.prompt_en, '\n\nREFERENCE:', t.ref_en, '\n\nEXPLANATION:', t.expl_en)
print('\nWRONG_WINDOW answers (consensus):', (graded.error_code == 'WRONG_WINDOW').sum())

Using Eurostat's monthly retail trade volume index (seasonally and calendar adjusted, 2021 = 100), which of these countries grew the most between January 2024 and September 2026: Spain, Finland, Ireland and Poland? Explain your choice of period. 

REFERENCE: Spain grew the most: +5.6% (3-month averages). Ranking: Spain +5.6%; Poland +5.3%; Ireland +4.7%; Finland +2.3%. 

EXPLANATION: 1) Period: a single month is noisy, so compare the average of January 2024-March 2024 with the average of July 2026-September 2026. 2) Compute: Spain 105.3 -> 111.2, i.e. +5.6%. 3) Trap: Endpoint-to-endpoint growth would put Poland first (+6.3%) because its January 2024 value (106.5) differs from its 3-month average (107.9): one atypical month drives the ranking. Data status: values used are flagged provisional ('p') for Spain, Ireland and Poland, so the result may be revised.

WRONG_WINDOW answers (consensus): 10


### 3.2 Percent vs percentage points

In [6]:
print(tasks[tasks.subtype == 'points_vs_percent'].iloc[0][['prompt_en', 'ref_en']].to_string())
print('PCT_VS_POINTS answers:', (graded.error_code == 'PCT_VS_POINTS').sum())

prompt_en    By how much did the unemployment rate (Eurosta...
ref_en       +0.5 percentage points (from 8.0% to 8.5%); in...
PCT_VS_POINTS answers: 8


### 3.3 Puzzles: the solver proves which answers break a rule

In [7]:
pa = answers[answers.task_type == 'logic_puzzle'].merge(tasks[['task_id', 'puzzle']], on='task_id')
en_names = {fr: en for en, fr in STOPS}
def broken(row):
    order = [x.strip() for x in row.answer.split('>')]
    if row.language == 'fr': order = [en_names[x] for x in order]
    return violated(json.loads(row.puzzle), order)
pa['broken_rules'] = pa.apply(broken, axis=1)
pa = pa.merge(graded[['answer_id', 'error_code']], on='answer_id')
print('answers breaking a rule (solver):', (pa.broken_rules.str.len() > 0).sum(), 'of', len(pa))
pd.crosstab(pa.broken_rules.str.len() > 0, pa.error_code, rownames=['solver: breaks a rule'])

answers breaking a rule (solver): 29 of 60


error_code,,ARITHMETIC,CONSTRAINT_IGNORED,FR_TERM,UNJUSTIFIED_STEP
solver: breaks a rule,,,,,
False,30,0,0,0,1
True,0,1,25,3,0


### 3.4 Claims confirmed with a stopped series (real: INSEE and Eurostat)

In [8]:
ins = pd.read_csv(PUB / 'insee_series.csv', dtype={'idbank': str})
print(ins.groupby('idbank').agg(title=('title_en', 'first'), last=('period', 'max')).to_string())
h = pd.read_csv(PUB / 'eurostat_hicp_stopped.csv'); print('\nprc_hicp_manr last period:', h.time.max(), '| updated', h.dataset_updated.iloc[0])
print('STALE_SOURCE answers:', (graded.error_code == 'STALE_SOURCE').sum())

                                                                                                             title     last
idbank                                                                                                                     
001688526                           ILO unemployment rate - Total - Metropolitan France - SA data - Stopped series  2026-Q1
001688527                                       ILO unemployment rate - Total - France excluding Mayotte - SA data  2026-Q3
001763852  Consumer price index - Base 2015 - All households - France - Overall excepting tobacco - Stopped series  2025-12
011814630       Consumer Price Index - Base 2025 - All households - France - COICOP classification: 00 - All items  2026-10

prc_hicp_manr last period: 2025-12 | updated 2026-02-06
STALE_SOURCE answers: 7


Real finding while building the tasks: three of the series we first reached for were stopped (INSEE CPI base 2015, INSEE unemployment for metropolitan France, Eurostat `prc_hicp_manr` frozen at 2025-12). A model, or an analyst, that only looks at the value would confirm an outdated figure.

### 3.5 French-specific errors

In [9]:
pd.read_csv(P / 'french_specific.csv')

,language,errors,french_specific,pct
0,en,43,0,0.0
1,fr,55,9,16.4


### 3.6 Real data issue: provisional and estimated values

In [10]:
pd.read_csv(P / 'retail_flags.csv')

,geo,provisional,estimated,values_total,first_flagged
0,AT,57,0,57,2022-01
1,BE,57,0,57,2022-01
2,DE,57,0,57,2022-01
3,ES,57,0,57,2022-01
4,FR,24,1,57,2024-09
5,IE,9,0,57,2026-01
6,NL,9,0,57,2026-01
7,PL,3,0,57,2026-07
8,SE,3,0,45,2026-07
9,FI,0,0,57,NaN


Four countries carry the `p` flag on **every** month since 2022 and France from 2024-09; the latest French value (2026-09) is `e`. A reference answer must say the result may be revised; an answer that calls it final is `partially_correct` (rule R4).

### 3.7 Real data issue: NUTS 2 labels and encoding

In [11]:
nl = pd.read_csv(PUB / 'eurostat_nights.csv')[['geo', 'geo_label']].drop_duplicates()
print(nl.to_string(index=False))
lab = nl.set_index('geo').geo_label
print('\nUTF-8 read as cp1252 ->', lab['FRJ2'].encode('utf-8').decode('cp1252'))
print('apostrophe in FRL0 label:', hex(ord([c for c in lab['FRL0'] if not c.isalnum() and c not in ' -'][0])))

 geo                  geo_label
FR10              Ile de France
FRE1         Nord-Pas de Calais
FRH0                   Bretagne
FRI1                  Aquitaine
FRJ2              Midi-Pyrénées
FRK2                Rhône-Alpes
FRL0 Provence-Alpes-Côte d’Azur
FRM0                      Corse

UTF-8 read as cp1252 -> Midi-PyrÃ©nÃ©es
apostrophe in FRL0 label: 0x2019


Labels are NUTS 2 statistical regions (Midi-Pyrénées, Aquitaine, Rhône-Alpes), not the 2016 administrative regions (Occitanie, Nouvelle-Aquitaine, Auvergne-Rhône-Alpes). Decoded in the wrong code page they become `Midi-PyrÃ©nÃ©es` (or `Midi-Pyr�n�es` in a Windows console); the typographic apostrophe (U+2019) in Provence-Alpes-Côte d’Azur breaks joins on a straight `'`.

### 3.8 Is the model worse in French? Check before reporting

In [12]:
raw = pd.read_csv(P / 'language_gap_raw.csv'); strata = pd.read_csv(P / 'language_gap_strata.csv')
std = pd.read_csv(P / 'language_gap_standardised.csv')
display(raw); display(strata); display(std)

,language,answers,pass_pct,pct_hard_items
0,en,120,64.2,24.2
1,fr,120,54.2,45.0


,difficulty,n_en,n_fr,pass_en,pass_fr,gap_fr_minus_en
0,1,47,30,89.4,90.0,0.6
1,2,44,36,59.1,61.1,2.0
2,3,29,54,31.0,29.6,-1.4


,language,standardised_pass_pct
0,en,59.1
1,fr,59.5


**False signal, not reported as a finding.** The raw pass rate is about 10 points lower in French, but French items are much more often rated difficulty 3. At equal difficulty the gap is within ±2 points, and the standardised pass rates are equal. What differs in French is the *kind* of error (French-specific formats), not the rate.

## 4. Do the reviewers agree? Guidelines v1 vs v2

In [13]:
agr = pd.read_csv(P / 'agreement.csv')
agr.groupby(['scope', 'guidelines'])[['observed_agreement', 'cohen_kappa']].mean().round(2)

observed_agreement  cohen_kappa
scope             guidelines                                 
all answers       v1                        0.77         0.59
                  v2                        0.92         0.87
grey-zone answers v1                        0.34         0.06
                  v2                        0.87        -0.04

In [14]:
pd.read_csv(P / 'confusion_r1_r3.csv')

,guidelines,r1_verdict,r3_correct,r3_partially_correct,r3_incorrect
0,v1,correct,126,8,0
1,v1,partially_correct,18,3,3
2,v1,incorrect,19,15,48
3,v2,correct,133,7,0
4,v2,partially_correct,5,27,4
5,v2,incorrect,0,6,58


In [15]:
pd.read_csv(P / 'time_per_review.csv')

,guidelines,task_type,reviews,median_s,p90_s
0,v1,claim_check,150,379.5,580.1
1,v1,logic_puzzle,180,270.0,428.3
2,v1,series_analysis,240,346.0,559.8
3,v1,summarise_chunk,150,289.0,457.4
4,v2,claim_check,150,330.5,552.3
5,v2,logic_puzzle,180,230.5,396.7
6,v2,series_analysis,240,309.5,467.9
7,v2,summarise_chunk,150,233.0,427.8


## 5. Answer key (opened last): were the planted events found?

In [16]:
key_items, key = ROOT / 'data' / '_hidden_truth_items.csv', ROOT / 'data' / '_hidden_truth.csv'
if key.exists():
    k = pd.read_csv(key_items).fillna({'planted_code': ''}).merge(graded[['answer_id', 'verdict', 'error_code']], on='answer_id')
    planted = k[k.planted_code != '']
    print(f"verdict matches truth: {(k.verdict == k.true_verdict).mean():.1%} | code found on planted errors: {(planted.error_code == planted.planted_code).mean():.1%}")
    display(pd.read_csv(key))
else:
    print('answer key not shipped: run python src/simulate_model.py to regenerate it locally')

verdict matches truth: 100.0% | code found on planted errors: 96.9%

,event_id,description,kind,n_answers
0,1,Wrong comparison window (growth from one atypi...,issue,10
1,2,Percent instead of percentage points,issue,9
2,3,Puzzle answer breaks one stated rule,issue,26
3,4,Claim confirmed with a stopped series / frozen...,issue,8
4,5,"French-specific errors: decimal comma, billion...",issue,9
5,6,REAL data issue: latest Eurostat values flagge...,issue,13
6,7,REAL data issue: NUTS 2 labels confused with c...,issue,5
7,8,FALSE SIGNAL: lower pass rate in French is a c...,false_signal,0


## 6. Recommendations
1. **Ask for the period before the answer.** For 'who grew the most', the reference compares 3-month averages, full years or the same month a year earlier, and says why.
2. **Check the source's status, not just its value**: stopped series, frozen datasets, `p`/`e` flags, NUTS level.
3. **Units are part of the answer**: points vs %, milliard vs billion, decimal comma.
4. **Prove puzzle answers** against every rule; an explanation that skips the elimination is not a full pass.
5. **Write rules for grey zones**: they caused most of the disagreement (kappa 0.59 → 0.87 with v2).
6. **Stratify before comparing languages**: the French gap here was a difficulty mix, not a weaker model.